# CGRA A1 — A.2 + A.3 (Convoy + Lemniscate)

Press **r** after clicking the canvas: wheels cube → cylinder → sphere.

**Voilà** only.



In [ ]:
%%html

<script src="https://is3l.isr.uc.pt/~pm/CGRA/JS/deecshader.js"></script>
<script src="https://is3l.isr.uc.pt/~pm/CGRA/JS/deecapp.js"></script>
<script src="https://is3l.isr.uc.pt/~pm/CGRA/JS/cgraobject.js"></script>
<script src="https://git.io/glm-js.min.js"></script>


<script id="my-vertex-shader" type="x-shader/x-vertex">
precision mediump float;
attribute  vec3 in_Position;
attribute  vec3 in_Color;
uniform mat4 MVP;
varying  vec3 ex_Color;
void main(void) {
    gl_Position = MVP * vec4(in_Position.x, in_Position.y, in_Position.z, 1.0);
    ex_Color = in_Color;
}
</script>


<script id="my-fragment-shader" type="x-shader/x-fragment">
precision mediump float;
varying  vec3 ex_Color;
void main(void) {
    gl_FragColor = vec4(ex_Color,1.0);
}
</script>


<div id="cgraStatus" style="color:#a00;font-family:monospace;"></div>
<script>
function cgraReady(){
  return (typeof DEECapp !== 'undefined')
      && (typeof DEECshader !== 'undefined')
      && (typeof CGRAobject !== 'undefined')
      && (typeof glm !== 'undefined');
}
function cgraBoot(startFn){
  var tries = 0;
  function go(){
    tries++;
    if (!cgraReady()){
      if (tries < 200){
        if (tries === 20){
          var el = document.getElementById('cgraStatus');
          if (el) el.textContent = 'Waiting for libraries (deecapp / cgraobject / glm)...';
        }
        setTimeout(go, 50);
        return;
      }
      var el2 = document.getElementById('cgraStatus');
      if (el2) el2.textContent = 'ERROR: libraries did not load. Check F12 console / VPN.';
      return;
    }
    var el3 = document.getElementById('cgraStatus');
    if (el3) el3.textContent = '';
    try { startFn(); }
    catch(err){
      console.error(err);
      var el4 = document.getElementById('cgraStatus');
      if (el4) el4.textContent = 'ERROR: ' + err.message;
      alert('CGRA error: ' + err.message);
    }
  }
  go();
}
</script>

<canvas id="myCanvasA23" width="700" height="500" style="border:2px solid #000000; background:#ddd;">
Error: Your browser does not support the HTML canvas tag.
</canvas>
<script>
cgraBoot(function(){
// ============================================================
// CGRA A1 — BASIC shapes (all extend CGRAobject)
// Keep this simple for oral defense.
// Color: smooth gradient from color1 to color2 along X.
// ============================================================

function mixColor(c1, c2, t) {
    // t from 0 to 1
    return [
        c1[0] * (1 - t) + c2[0] * t,
        c1[1] * (1 - t) + c2[1] * t,
        c1[2] * (1 - t) + c2[2] * t
    ];
}

function pushTri(verts, cols, ax, ay, az, bx, by, bz, cx, cy, cz, c1, c2) {
    // one triangle = 3 vertices
    var pts = [[ax, ay, az], [bx, by, bz], [cx, cy, cz]];
    for (var i = 0; i < 3; i++) {
        verts.push(pts[i][0], pts[i][1], pts[i][2]);
        // gradient by X: -1..1 mapped to 0..1
        var t = (pts[i][0] + 1.0) * 0.5;
        if (t < 0) t = 0;
        if (t > 1) t = 1;
        var col = mixColor(c1, c2, t);
        cols.push(col[0], col[1], col[2]);
    }
}

function uploadBuffers(obj, verts, cols) {
    obj.numvertices = verts.length / 3;
    obj.vertexbuffer = obj.gl.createBuffer();
    obj.gl.bindBuffer(obj.gl.ARRAY_BUFFER, obj.vertexbuffer);
    obj.gl.bufferData(obj.gl.ARRAY_BUFFER, new Float32Array(verts), obj.gl.STATIC_DRAW);

    obj.colorbuffer = obj.gl.createBuffer();
    obj.gl.bindBuffer(obj.gl.ARRAY_BUFFER, obj.colorbuffer);
    obj.gl.bufferData(obj.gl.ARRAY_BUFFER, new Float32Array(cols), obj.gl.STATIC_DRAW);
}

// ---------- CUBE: 6 faces * 2 triangles * 3 verts = 36 ----------
class Cube extends CGRAobject {
    constructor(glcontext, color1, color2) {
        super(glcontext);
        var c1 = color1 || [1, 0, 0];
        var c2 = color2 || [0, 0, 1];
        var verts = [];
        var cols = [];

        // unit cube corners from -1 to 1
        // faces: front, back, left, right, top, bottom
        // each face = 2 triangles
        var f = [
            // front z=+1
            [-1, -1, 1, 1, -1, 1, 1, 1, 1, -1, -1, 1, 1, 1, 1, -1, 1, 1],
            // back z=-1
            [1, -1, -1, -1, -1, -1, -1, 1, -1, 1, -1, -1, -1, 1, -1, 1, 1, -1],
            // left x=-1
            [-1, -1, -1, -1, -1, 1, -1, 1, 1, -1, -1, -1, -1, 1, 1, -1, 1, -1],
            // right x=+1
            [1, -1, 1, 1, -1, -1, 1, 1, -1, 1, -1, 1, 1, 1, -1, 1, 1, 1],
            // top y=+1
            [-1, 1, 1, 1, 1, 1, 1, 1, -1, -1, 1, 1, 1, 1, -1, -1, 1, -1],
            // bottom y=-1
            [-1, -1, -1, 1, -1, -1, 1, -1, 1, -1, -1, -1, 1, -1, 1, -1, -1, 1]
        ];

        for (var fi = 0; fi < f.length; fi++) {
            var a = f[fi];
            // two triangles already listed as 6 verts
            pushTri(verts, cols, a[0], a[1], a[2], a[3], a[4], a[5], a[6], a[7], a[8], c1, c2);
            pushTri(verts, cols, a[9], a[10], a[11], a[12], a[13], a[14], a[15], a[16], a[17], c1, c2);
        }
        uploadBuffers(this, verts, cols);
    }
}

// ---------- DISC: center + ring (>=10) ----------
class Disc extends CGRAobject {
    constructor(glcontext, color1, color2, n) {
        super(glcontext);
        var c1 = color1 || [1, 1, 0];
        var c2 = color2 || [1, 0.5, 0];
        n = n || 12; // at least 10
        var verts = [];
        var cols = [];
        var r = 1.0;
        for (var i = 0; i < n; i++) {
            var a0 = (i / n) * 2 * Math.PI;
            var a1 = ((i + 1) / n) * 2 * Math.PI;
            var x0 = r * Math.cos(a0), z0 = r * Math.sin(a0);
            var x1 = r * Math.cos(a1), z1 = r * Math.sin(a1);
            // triangle: center, p0, p1  (in XZ plane, y=0)
            pushTri(verts, cols, 0, 0, 0, x0, 0, z0, x1, 0, z1, c1, c2);
        }
        uploadBuffers(this, verts, cols);
    }
}

// ---------- CONE: apex + base circle (>=10) ----------
class Cone extends CGRAobject {
    constructor(glcontext, color1, color2, n) {
        super(glcontext);
        var c1 = color1 || [0.8, 0.4, 0.1];
        var c2 = color2 || [0.3, 0.15, 0.05];
        n = n || 12;
        var verts = [];
        var cols = [];
        var r = 1.0;
        var apexY = 1.0;
        var baseY = -1.0;
        for (var i = 0; i < n; i++) {
            var a0 = (i / n) * 2 * Math.PI;
            var a1 = ((i + 1) / n) * 2 * Math.PI;
            var x0 = r * Math.cos(a0), z0 = r * Math.sin(a0);
            var x1 = r * Math.cos(a1), z1 = r * Math.sin(a1);
            // side triangle
            pushTri(verts, cols, 0, apexY, 0, x0, baseY, z0, x1, baseY, z1, c1, c2);
            // base triangle (optional close)
            pushTri(verts, cols, 0, baseY, 0, x1, baseY, z1, x0, baseY, z0, c1, c2);
        }
        uploadBuffers(this, verts, cols);
    }
}

// ---------- CYLINDER: two circles (>=10), quads -> 2 tris ----------
class Cylinder extends CGRAobject {
    constructor(glcontext, color1, color2, n) {
        super(glcontext);
        var c1 = color1 || [0.6, 0.6, 0.6];
        var c2 = color2 || [0.2, 0.2, 0.2];
        n = n || 12;
        var verts = [];
        var cols = [];
        var r = 1.0;
        var yTop = 1.0, yBot = -1.0;
        for (var i = 0; i < n; i++) {
            var a0 = (i / n) * 2 * Math.PI;
            var a1 = ((i + 1) / n) * 2 * Math.PI;
            var x0 = r * Math.cos(a0), z0 = r * Math.sin(a0);
            var x1 = r * Math.cos(a1), z1 = r * Math.sin(a1);
            // side: two triangles
            pushTri(verts, cols, x0, yBot, z0, x1, yBot, z1, x1, yTop, z1, c1, c2);
            pushTri(verts, cols, x0, yBot, z0, x1, yTop, z1, x0, yTop, z0, c1, c2);
            // top cap
            pushTri(verts, cols, 0, yTop, 0, x0, yTop, z0, x1, yTop, z1, c1, c2);
            // bottom cap
            pushTri(verts, cols, 0, yBot, 0, x1, yBot, z1, x0, yBot, z0, c1, c2);
        }
        uploadBuffers(this, verts, cols);
    }
}

// ---------- SPHERE: latitude strips (>=10) * rings (>=10) ----------
class Sphere extends CGRAobject {
    constructor(glcontext, color1, color2, stacks, slices) {
        super(glcontext);
        var c1 = color1 || [0.2, 0.6, 1.0];
        var c2 = color2 || [1.0, 0.2, 0.4];
        stacks = stacks || 12; // latitude strips
        slices = slices || 12; // vertices per ring
        var verts = [];
        var cols = [];
        var R = 1.0;

        for (var i = 0; i < stacks; i++) {
            // elevation from -PI/2 to +PI/2
            var lat0 = -Math.PI / 2 + (i / stacks) * Math.PI;
            var lat1 = -Math.PI / 2 + ((i + 1) / stacks) * Math.PI;
            var y0 = R * Math.sin(lat0);
            var y1 = R * Math.sin(lat1);
            var r0 = R * Math.cos(lat0);
            var r1 = R * Math.cos(lat1);

            for (var j = 0; j < slices; j++) {
                var a0 = (j / slices) * 2 * Math.PI;
                var a1 = ((j + 1) / slices) * 2 * Math.PI;

                var x00 = r0 * Math.cos(a0), z00 = r0 * Math.sin(a0);
                var x01 = r0 * Math.cos(a1), z01 = r0 * Math.sin(a1);
                var x10 = r1 * Math.cos(a0), z10 = r1 * Math.sin(a0);
                var x11 = r1 * Math.cos(a1), z11 = r1 * Math.sin(a1);

                // two triangles between rings
                pushTri(verts, cols, x00, y0, z00, x01, y0, z01, x11, y1, z11, c1, c2);
                pushTri(verts, cols, x00, y0, z00, x11, y1, z11, x10, y1, z10, c1, c2);
            }
        }
        uploadBuffers(this, verts, cols);
    }
}

function T(x,y,z){ return glm.translate(glm.mat4(1.0), glm.vec3(x,y,z)); }
function R(deg, ax,ay,az){ return glm.rotate(glm.mat4(1.0), glm.radians(deg), glm.vec3(ax,ay,az)); }
function Su(s){ return glm.mat4(glm.mat3(s)); } // uniform scale (professor style)
function mul(a,b,c,d){
  var m = a['*'](b);
  if (c) m = m['*'](c);
  if (d) m = m['*'](d);
  return m;
}

function lemniPos(omega, a, b){
    var s=Math.sin(omega), c=Math.cos(omega);
    var den=1.0+s*s;
    return {x:a*c/den, z:b*s*c/den};
}
function lemniDeriv(omega, a, b, eps){
    var s=Math.sin(omega), c=Math.cos(omega);
    var den=1.0+s*s, den2=den*den;
    var dx=-a*eps*s*(2.0+c*c)/den2;
    var numer=b*(Math.cos(2.0*omega)*den-(s*c)*(2.0*s*c));
    var dz=eps*numer/den2;
    return {dx:dx, dz:dz};
}
class myapp extends DEECapp{
    counter=0;
    wheelMode=0;
    initialize(){
        var fragsrc=document.getElementById("my-fragment-shader").text;
        var vertsrc=document.getElementById("my-vertex-shader").text;
        this.shaderprog=new DEECshader(this.gl);
        this.shaderprog.srcShaders(vertsrc,fragsrc);
        this.gl.enable(this.gl.DEPTH_TEST);
        this.gl.clearColor(0.7,0.85,0.95,1.0);
        this.cube=new Cube(this.gl,[0.55,0.32,0.12],[0.25,0.12,0.05]);
        this.cone=new Cone(this.gl,[0.7,0.35,0.1],[0.3,0.15,0.05],12);
        this.cyl=new Cylinder(this.gl,[0.4,0.4,0.4],[0.15,0.15,0.15],12);
        this.sph=new Sphere(this.gl,[0.3,0.3,0.35],[0.1,0.1,0.12],12,12);
        this.dark=new Cube(this.gl,[0.1,0.1,0.1],[0.05,0.05,0.05]);
        var all=[this.cube,this.cone,this.cyl,this.sph,this.dark];
        for(var i=0;i<all.length;i++) all[i].setShader(this.shaderprog);
        this.a=0.55; this.b=0.35; this.eps=0.4;
        this.theta=[0,-0.6,-1.2,-1.8];
        this.start=Date.now();
        window.addEventListener("keydown",(e)=>{
            if(e.key.toLowerCase()==='r'){ this.wheelMode=(this.wheelMode+1)%3; console.log('wheels',this.wheelMode); }
        },false);
        console.log('A2/A3 convoy ready');
    }
    draw(obj,M){
        obj.setModelTransformation(M);
        obj.drawit(); // NDC / Objects style
    }
    render(){
        this.gl.clear(this.gl.COLOR_BUFFER_BIT|this.gl.DEPTH_BUFFER_BIT);
        var t=(Date.now()-this.start)*0.001;
        for(var i=0;i<4;i++){
            var w=this.eps*t+this.theta[i];
            var p=lemniPos(w,this.a,this.b);
            var d=lemniDeriv(w,this.a,this.b,this.eps);
            var yaw=Math.atan2(d.dx,d.dz)*180.0/Math.PI;
            var world=mul(T(p.x, 0.0, p.z*0.01), R(yaw,0,1,0)); // flatten z a bit for 2.5D view
            // better: show path in x-y plane for simple view
            world=mul(T(p.x, p.z, 0.0), R(this.counter*0.2 + i*10, 0.2,1,0.1));
            var body = (i===0) ? Su(0.08) : Su(0.06 + 0.01*i);
            this.draw(this.cube, mul(world, body));
            if(i===0) this.draw(this.cone, mul(world, T(0.12,0,0), R(90,0,0,1), Su(0.05)));
            var offs=[[-0.05,-0.05],[0.05,-0.05],[-0.05,0.05],[0.05,0.05]];
            for(var k=0;k<4;k++){
                var o=offs[k];
                var Wm=mul(world, T(o[0], o[1], 0.0), Su(0.025));
                if(this.wheelMode===0) this.draw(this.dark, Wm);
                else if(this.wheelMode===1) this.draw(this.cyl, Wm);
                else this.draw(this.sph, Wm);
            }
        }
        this.counter++;
    }
}
var app=new myapp('myCanvasA23');
app.run();

});
</script>

